# Clase 11: Aprendizaje por Refuerzo — Fundamentos y Gymnasium

**Machine Learning (RIA-805) — Bloque 2 — Factibilidad Técnico-Financiera**

**Docente:** Marco Capio  
**Fecha:** Semana 3 del bloque comprimido

**Objetivo:** Comprender los fundamentos del Aprendizaje por Refuerzo (agente, entorno, estado, acción, recompensa, política) y explorar entornos de Gymnasium como CartPole.

**Conexión con la práctica:** En P2-6 y P2-7 explorarán Gymnasium y crearán un entorno personalizado. Hoy sentamos las bases.

## ¿Dónde estamos?

El Bloque 2 comenzó con factibilidad técnico-financiera y continuó con métricas para datos desbalanceados, análisis de costos, ajuste del umbral (*threshold tuning*) y optimización de hiperparámetros con Optuna. Hoy entramos a un paradigma distinto: el Aprendizaje por Refuerzo, donde un agente aprende mediante la interacción con un entorno.

> **Hasta ahora: supervisado y no supervisado. Hoy: RL.**

## ¿Qué es el Aprendizaje por Refuerzo?

El **Aprendizaje por Refuerzo** (*Reinforcement Learning*, RL) es un paradigma de aprendizaje en el que un agente toma acciones en un entorno y recibe señales de recompensa. A partir de esa experiencia, aprende a elegir acciones que mejoren los resultados futuros.

| Paradigma | Datos | Objetivo | Ejemplo |
|---|---|---|---|
| Supervisado | Ejemplos con etiquetas o respuestas correctas | Predecir la etiqueta o el valor esperado | Clasificar imágenes de gatos y perros |
| No supervisado | Datos sin etiquetas | Encontrar estructura, grupos o representaciones | Agrupar clientes por patrones de compra |
| Aprendizaje por refuerzo | Experiencia de interacción: estados, acciones y recompensas | Aprender una política que maximice el retorno | Enseñar a un robot a mantener el equilibrio |

> **El agente aprende de la experiencia, no de un dataset etiquetado.**

## Elementos del RL

- **Agente:** quien toma decisiones; por ejemplo, un robot o un programa.
- **Entorno:** el mundo con el que interactúa el agente y que responde a sus acciones.
- **Estado:** descripción de la situación actual que el agente observa.
- **Acción:** decisión que el agente puede ejecutar en un estado.
- **Recompensa:** señal numérica que indica qué tan conveniente fue un resultado inmediato.
- **Política:** estrategia que indica qué acción elegir en cada estado; puede ser determinista o probabilística.
- **Valor:** estimación de la recompensa acumulada futura que puede obtenerse desde un estado (o al tomar una acción en él).

## El ciclo de interacción

```text
                 acción aₜ
        ┌────────────────────────►┐
        │                         │
   ┌────┴────┐               ┌────▼─────┐
   │ Agente  │               │ Entorno  │
   └────▲────┘               └────┬─────┘
        └─────────────────────────┘
             estado sₜ₊₁, recompensa rₜ₊₁
```

El ciclo se repite durante la interacción:

1. El agente observa el estado actual.
2. Su política selecciona una acción.
3. El entorno ejecuta la acción y cambia de estado.
4. El agente recibe el nuevo estado y una recompensa; usa la experiencia para mejorar su política.

## Episodios y recompensa acumulada

Un **episodio** es una secuencia de interacciones que comienza en un estado inicial y termina cuando se alcanza una condición terminal o un límite de tiempo. En **CartPole**, el episodio termina si el poste cae o si se alcanza el máximo de pasos permitido.

Como las recompensas futuras pueden ser menos seguras o menos valiosas, se usa un factor de descuento `γ` entre 0 y 1:

$$G_t = \sum_{k=0}^{T-t-1} \gamma^k r_{t+k+1}$$

Aquí `G_t` es el retorno desde el tiempo `t`, `r` representa cada recompensa y `T` el final del episodio. Un `γ` cercano a 1 da más importancia a recompensas lejanas; uno menor prioriza las inmediatas.

> **Objetivo: maximizar recompensa acumulada promedio.**

## Exploración vs. explotación

El agente debe equilibrar **exploración** (probar acciones para descubrir sus resultados) y **explotación** (usar la mejor acción que conoce). Por ejemplo, un robot que busca una estación de carga puede dirigirse a una estación conocida o explorar una ruta nueva que quizá sea más corta.

Estrategias comunes:

- **ε-greedy:** con probabilidad `ε` se prueba una acción aleatoria; de lo contrario, se elige la acción estimada como mejor.
- **Softmax:** asigna probabilidades a las acciones según sus valores estimados; la temperatura controla cuán variadas son las elecciones.
- **Entropía:** mide la aleatoriedad de una política; algunos métodos la favorecen durante el entrenamiento para evitar que el agente deje de explorar demasiado pronto.

## Tipos de RL

| Distinción | Tipo | Descripción | Ejemplo |
|---|---|---|---|
| Espacio de acciones | Discreto | Acciones de un conjunto finito | Mover a la izquierda o derecha |
| Espacio de acciones | Continuo | Acciones con valores dentro de un rango | Elegir el ángulo de giro de un motor |
| Política de aprendizaje | On-policy | Aprende con la política que está usando actualmente | SARSA; PPO suele entrenarse de forma on-policy |
| Política de aprendizaje | Off-policy | Aprende de datos que pueden provenir de otra política o de una política de exploración | Q-learning; DQN |
| Uso de un modelo | Model-free | Aprende sin conocer de antemano la dinámica del entorno | DQN aprende valores a partir de experiencia |
| Uso de un modelo | Model-based | Usa o aprende un modelo de cómo evoluciona el entorno para planificar | Planificación con un simulador conocido |

## Gymnasium: la librería estándar

**Gymnasium** es una biblioteca de código abierto para definir y usar entornos de Aprendizaje por Refuerzo. Es la sucesora mantenida por la comunidad de OpenAI Gym. Incluye entornos predefinidos y una API estándar para reiniciar un entorno, ejecutar acciones, observar resultados y liberar recursos.

Si aún no está instalada en el entorno de Python seleccionado para Jupyter, puedes instalarla ejecutando:

```bash
pip install gymnasium
```

En este notebook usaremos **CartPole-v1**, un problema clásico en el que se intenta mantener un poste en equilibrio sobre un carrito.

In [ ]:
# Importamos Gymnasium, arreglos numéricos, gráficos, utilidades de rutas y tiempo.
import os
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

# En Jupyter, mostramos los gráficos dentro del notebook.
%matplotlib inline

# Fijamos la semilla del generador de acciones para que las muestras aleatorias
# sean reproducibles al ejecutar el notebook de nuevo.
SEED = 42
print(f"Versión de Gymnasium: {gym.__version__}")

## La API de Gymnasium

| Método | Función |
|---|---|
| `reset(seed=..., options=...)` | Inicia un episodio y devuelve `(observation, info)`. La semilla permite reproducir la inicialización. |
| `step(action)` | Ejecuta una acción y devuelve `(observation, reward, terminated, truncated, info)`. |
| `render()` | Produce una representación visual, si el entorno se creó con un modo de renderizado compatible. |
| `close()` | Libera los recursos del entorno. |

`terminated` indica que se alcanzó un estado terminal definido por la tarea; `truncated` indica que el episodio se interrumpió por un límite externo, como el máximo de pasos. Conviene detener el episodio cuando cualquiera de los dos sea verdadero.

| Atributo | Función |
|---|---|
| `action_space` | Define las acciones permitidas y permite muestrearlas con `.sample()`. |
| `observation_space` | Describe la forma y el rango de las observaciones que puede devolver el entorno. |

In [ ]:
# Creamos CartPole-v1 sin renderizado para que funcione también en entornos sin pantalla.
env = gym.make("CartPole-v1")

# Discrete(2) representa dos acciones; Box(4,) describe cuatro valores continuos observados.
print("Espacio de acciones:", env.action_space)
print("Espacio de observaciones:", env.observation_space)

# Para visualizarlo en local, se puede crear con render_mode="human" si está disponible.
# No llamamos a env.render() aquí para mantener compatibilidad con entornos sin pantalla.

## Explorar el entorno

Vamos a resetear el entorno y ver el estado inicial. La observación de CartPole es un vector numérico con cuatro medidas del sistema.

In [ ]:
# Iniciamos el episodio con una semilla fija para repetir el estado inicial.
state, info = env.reset(seed=SEED)
print("Estado inicial:", state)
print("Información adicional:", info)

# Las dimensiones son: posición del carrito, velocidad del carrito,
# ángulo del poste y velocidad angular del poste.

## Ejecutar acciones aleatorias

Vamos a ejecutar hasta 5 acciones aleatorias y ver cómo cambian el estado y la recompensa. Aunque la secuencia de acciones sea aleatoria, fijamos la semilla del espacio de acciones.

In [ ]:
# Sembramos el generador de acciones y avanzamos como máximo cinco pasos.
env.action_space.seed(SEED)
for paso in range(5):
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    print(
        f"Paso {paso + 1}: acción={action}, estado={state}, "
        f"recompensa={reward}, terminated={terminated}, truncated={truncated}"
    )

    # Un episodio terminado no debe recibir más llamadas a step() hasta hacer reset().
    if terminated or truncated:
        print("El episodio terminó antes de completar los cinco pasos.")
        break

## Ejecutar un episodio completo

Ahora ejecutamos un episodio completo con acciones aleatorias hasta que termine. En CartPole-v1, normalmente se recibe una recompensa de 1 por cada paso que el poste se mantiene en equilibrio.

In [ ]:
# Reiniciamos con una semilla fija y acumulamos la recompensa de todo el episodio.
state, info = env.reset(seed=SEED)
total_reward = 0.0
steps = 0
terminated = False
truncated = False

while not (terminated or truncated):
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1

print(f"Recompensa acumulada: {total_reward:.0f}")
print(f"Número de pasos: {steps}")
# Con acciones aleatorias, el episodio suele durar pocos pasos frente al máximo de 500.

## Ejecutar múltiples episodios

Vamos a ejecutar 10 episodios con acciones aleatorias y ver la recompensa acumulada de cada uno. Varias ejecuciones ayudan a observar que una política aleatoria puede producir resultados distintos.

In [ ]:
# Ejecutamos diez episodios con estados iniciales reproducibles y guardamos sus retornos.
num_episodes = 10
episode_rewards = []

for episode in range(num_episodes):
    state, info = env.reset(seed=SEED + episode)
    total_reward = 0.0
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action = env.action_space.sample()
        state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

    episode_rewards.append(total_reward)
    print(f"Episodio {episode + 1}: recompensa acumulada = {total_reward:.0f}")

print(f"\nPromedio: {np.mean(episode_rewards):.2f}")
print(f"Máximo: {np.max(episode_rewards):.0f}")
print(f"Mínimo: {np.min(episode_rewards):.0f}")

## Graficar recompensas

La gráfica muestra la recompensa de cada episodio. Con acciones aleatorias, la recompensa suele ser baja y variable: sin una política que use el estado para escoger acciones, el poste generalmente pierde el equilibrio pronto.

In [ ]:
# Graficamos los retornos y creamos la carpeta de destino si todavía no existe.
plt.figure(figsize=(9, 4))
plt.plot(range(1, num_episodes + 1), episode_rewards, marker="o")
plt.xlabel("Episodio")
plt.ylabel("Recompensa acumulada")
plt.title("CartPole-v1: política de acciones aleatorias")
plt.xticks(range(1, num_episodes + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()

os.makedirs("reports/figures", exist_ok=True)
figure_path = "reports/figures/cartpole_aleatorio.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Gráfico guardado en: {figure_path}")

## Comparación con un agente entrenado

Un agente entrenado puede obtener recompensas mucho más altas, cercanas a 500 en CartPole-v1, dependiendo del entrenamiento y la evaluación. Con acciones aleatorias, la recompensa suele quedar muy por debajo —a menudo apenas llega a 20–30 pasos— porque las decisiones no consideran el estado.

Estas cifras son orientativas: los resultados dependen de la semilla, la implementación y el criterio de evaluación. En la próxima clase entrenaremos un agente real.

## Crear un entorno personalizado

Gymnasium permite crear entornos propios heredando de `gym.Env`. Vamos a modelar un agente en una línea que puede moverse a izquierda o derecha y debe llegar a una meta en los extremos.

Los métodos principales son:

- `__init__`: define los espacios de acciones y observaciones, además del estado interno.
- `reset`: inicia un episodio y devuelve la observación inicial junto con un diccionario `info`.
- `step`: aplica una acción y devuelve observación, recompensa, `terminated`, `truncated` e `info`.

Usaremos una observación vectorial de una dimensión para que coincida con el espacio `Box` declarado.

In [ ]:
class MiEntorno(gym.Env):
    """Entorno lineal mínimo con acciones para moverse a izquierda o derecha."""

    def __init__(self):
        """Inicializa los espacios y la posición del agente."""
        super().__init__()
        self.action_space = gym.spaces.Discrete(2)
        self.observation_space = gym.spaces.Box(
            low=-5.0, high=5.0, shape=(1,), dtype=np.float32
        )
        self.state = 0.0

    def reset(self, *, seed=None, options=None):
        """Reinicia el agente en el centro y devuelve observación e información."""
        super().reset(seed=seed)
        self.state = 0.0
        observation = np.array([self.state], dtype=np.float32)
        return observation, {}

    def step(self, action):
        """Aplica una acción, calcula su recompensa y señala si llegó a la meta."""
        if not self.action_space.contains(action):
            raise ValueError(f"Acción inválida: {action}")

        if action == 0:
            self.state -= 1.0
        else:
            self.state += 1.0

        reward = -abs(self.state)
        terminated = abs(self.state) >= 5.0
        truncated = False
        observation = np.array([self.state], dtype=np.float32)
        return observation, float(reward), terminated, truncated, {}

In [ ]:
# Probamos el entorno lineal y mostramos hasta diez acciones aleatorias.
mi_entorno = MiEntorno()
state, info = mi_entorno.reset(seed=SEED)
mi_entorno.action_space.seed(SEED)

for paso in range(10):
    action = mi_entorno.action_space.sample()
    state, reward, terminated, truncated, info = mi_entorno.step(action)
    print(
        f"Paso {paso + 1}: acción={action}, estado={state}, "
        f"recompensa={reward:.1f}, terminated={terminated}"
    )

    # Al alcanzar un extremo, iniciamos otro episodio antes de continuar la demostración.
    if terminated or truncated:
        print("Se alcanzó un extremo; reiniciamos el entorno.")
        state, info = mi_entorno.reset()

mi_entorno.close()

## Aplicaciones en robótica

| Área | Ejemplo de aplicación |
|---|---|
| Locomoción | Aprender a caminar, correr o mantener el equilibrio en distintas superficies |
| Manipulación | Agarrar, orientar y colocar objetos con un brazo robótico |
| Navegación | Planificar una ruta y evitar obstáculos en un almacén |
| Control | Ajustar motores para seguir una trayectoria o estabilizar un dron |
| Juegos | Aprender estrategias para resolver juegos de mesa o videojuegos |

**Ventaja:** no requiere un dataset etiquetado con la acción correcta para cada situación.  
**Desventaja:** entrenar puede ser lento y costoso, especialmente en robots reales.

Una estrategia habitual es **sim-to-real**: entrenar primero en simulación, donde los intentos son más baratos y seguros, y luego adaptar y validar cuidadosamente el comportamiento en el sistema real. La simulación no elimina por completo la diferencia entre ambos mundos.

## Conclusiones

1. **RL** aprende mediante la interacción y las recompensas, en lugar de depender de etiquetas para cada acción.
2. Sus elementos clave son el agente, el entorno, el estado, la acción, la recompensa, la política y el valor.
3. El ciclo agente-entorno genera experiencia; el objetivo es maximizar el retorno acumulado.
4. **Gymnasium** ofrece una API común para crear entornos, reiniciarlos, ejecutar pasos y finalizar episodios.
5. **CartPole** muestra cómo observar estados, seleccionar acciones y medir el desempeño de una política.

**Próxima clase:** entrenamiento de un agente con PPO/DQN y Stable-Baselines3.

In [ ]:
# Cerramos CartPole para liberar los recursos asociados al entorno.
env.close()
print("Clase 11 completada. Revisa reports/figures/cartpole_aleatorio.png")